purpose of playground is to integrate and test rewritten functions

In [10]:
import plotly.graph_objects as go
import plotly.express as px
import utils_for_plotly_rewrites
import nist_codes
import prep_utils
import csv
import helper_utils
import numpy as np
import pandas as pd
import io
from scipy.signal import find_peaks

In [11]:
HE_SHEET = "he test.xlsx"
H_SHEET = "h test with headers.xlsx"
NIST_SHEET = "oxygen nist 2.xlsx"
HE_INC_INT_SHEET = "he test w increased int vals.xlsx"

detection_col = '_raw_int'
int_col = '_adj_int'

In [12]:
def set_y_lim(graph_type, show_peak_labels):
    if graph_type == 'scatter' or show_peak_labels is True:
        if prep_utils.Y_MAX > 230:
            y_max = utils_for_plotly_rewrites.round_to_multiple(prep_utils.Y_MAX, 100)
        else:
            y_max = utils_for_plotly_rewrites.round_to_multiple(prep_utils.Y_MAX, 50)
    else:
        y_max = utils_for_plotly_rewrites.round_to_multiple(prep_utils.Y_MAX, 50)
    print('(set_y_lim)', y_max)
    return y_max

In [13]:
def set_major_y_ticks(y_max):
    if 150 <= y_max <= 350:
        major_y_tick = 50
    elif 400 <= y_max <= 800:
        major_y_tick = 100
    elif 900 <= y_max <= 1400:
        major_y_tick = 200
    elif 1500 <= y_max <= 3500:
        major_y_tick = 500
    elif 4000 <= y_max <= 8000:
        major_y_tick = 1000

    return major_y_tick

In [14]:
def axis_labels(
    graph_type,
    show_peak_labels,
    plot_title=None,
    x_title=utils_for_plotly_rewrites.X_TITLE,
    y_title=utils_for_plotly_rewrites.Y_TITLE,
    x_range=[utils_for_plotly_rewrites.X_MIN, utils_for_plotly_rewrites.X_MAX],
    show_grid=True,
    bg_colour = utils_for_plotly_rewrites.BG,
    text_colour=utils_for_plotly_rewrites.COLOUR,
    grid_color='#222222',
    grid_width=0.1,
    grid_dash='dot',
    line_color='#333333',
    line_width=0.2,
    major_ticks=50, 
    fig_width=utils_for_plotly_rewrites.fig_size[0] * 70, # will have to keep adjusting
    fig_height=utils_for_plotly_rewrites.fig_size[1] * 97,
    tick_padding = 7,
):

    y_max = set_y_lim(graph_type=graph_type, show_peak_labels=show_peak_labels)
    y_range = [0, y_max]

    dtick_y = set_major_y_ticks(y_max=y_max)

    layout_config = dict(
        plot_bgcolor=bg_colour,
        paper_bgcolor=bg_colour,
        font=dict(color=text_colour),
        width=fig_width,
        height=fig_height,
        xaxis=dict(
            title=x_title,
            range=x_range,
            showgrid=show_grid,
            gridcolor=grid_color,
            gridwidth=grid_width,
            griddash=grid_dash,
            showline=True,
            linecolor=line_color,
            linewidth=line_width,
            mirror=True,
            tickfont=dict(color=text_colour),
            dtick=major_ticks,
            ticklabelstandoff = tick_padding
        ),
        
        yaxis=dict(
            title=y_title,
            range=y_range, 
            showgrid=show_grid,
            gridcolor=grid_color,
            gridwidth=grid_width,
            griddash=grid_dash,
            showline=True,
            linecolor=line_color,
            linewidth=line_width,
            mirror=True,
            tickfont=dict(color=text_colour),
            dtick=dtick_y,
            ticklabelstandoff = tick_padding
        ),

        title=dict(
            text=plot_title,
            font=dict(color=text_colour) if plot_title else None 
        ) if plot_title else None
    )

    if not show_grid:
        layout_config['xaxis']['showgrid'] = False
        layout_config['yaxis']['showgrid'] = False

    return layout_config


In [15]:
def peak_labels(data_df, show_label_colour, has_any_nist):

    if has_any_nist is False:
        peaks, _ = find_peaks(data_df[helper_utils.INT_col], prominence=utils_for_plotly_rewrites.dynamic_prominence(helper_utils.DEFAULT_PROM_PERC, prep_utils.int_range)) 
        int_col = helper_utils.INT_col
    else:
        peaks, _ = find_peaks(data_df['_raw_int'], prominence=helper_utils.DEFAULT_PROM_PERC)
        int_col = ['_raw_int']

    labels = []
    for peak_index in peaks:
        row = data_df.iloc[peak_index]
        wavelength = row[helper_utils.wl_col]
        #wl_labels = float(row)
        base_rgb=utils_for_plotly_rewrites.rgb(wavelength)
        final_int_scale = 1.0
        colour_rgb=utils_for_plotly_rewrites.colored_rgb(base_rgb, final_int_scale)
        color_str = utils_for_plotly_rewrites.colours(base_rgb, final_int_scale)
        if show_label_colour is False:
            font_colour = utils_for_plotly_rewrites.COLOUR
        else:
            font_colour = color_str
        label_config = dict(
            x=row[helper_utils.wl_col],
            y=row[int_col],
            text=f"{row[helper_utils.wl_col]:.2f} nm",
            showarrow=False,
            yshift=15,
            opacity=1,
            font=dict(color=font_colour)
        )
        labels.append(label_config)

    return labels

In [16]:
def bar_iter(data_df, show_peak_labels, show_label_colour, scale_by_int):
    fig_bar = go.Figure();

    if show_peak_labels is True:
        fig_bar.update_layout(annotations = peak_labels(data_df=data_df, show_label_colour=show_label_colour, has_any_nist=False))

    bar_colors = []
    for index, row in data_df.iterrows():
        wavelength = row[helper_utils.wl_col]
        normalized_intensity = row['Norm_Int']

        base_rgb=utils_for_plotly_rewrites.rgb(wavelength)

        if scale_by_int is True:
            final_intensity_scale=utils_for_plotly_rewrites.final_scale(utils_for_plotly_rewrites.min_bright, normalized_intensity)
        else:
            final_intensity_scale = 1.0

        colored_rgb=utils_for_plotly_rewrites.colored_rgb(base_rgb, final_intensity_scale)

        r, g, b = int(colored_rgb[0]), int(colored_rgb[1]), int(colored_rgb[2])
        bar_colors.append(f"rgba({r}, {g}, {b}, {1.0})")

    
    fig_bar.add_trace(go.Bar(
        x=data_df[helper_utils.wl_col],
        y=data_df[helper_utils.INT_col],
        marker=dict(color=bar_colors), # Use marker dict for color
        marker_line_color=bar_colors,
        marker_line_width=0,
    ));

    fig_bar.update_layout(axis_labels(graph_type='bar', show_peak_labels=show_peak_labels))
    fig_bar.show()

In [17]:
def line_iter(data_df, show_peak_labels, show_label_colour, scale_by_int):
    line_fig = go.Figure()

    if show_peak_labels is True:
        line_fig.update_layout(annotations = peak_labels(data_df=data_df, show_label_colour=show_label_colour, has_any_nist=False))


    for i in range(len(data_df) - 1):
        wavelength_start = float(data_df.iloc[i][helper_utils.wl_col])
        wavelength_end = float(data_df.iloc[i+1][helper_utils.wl_col])
        int_factor = float(data_df.iloc[i]['Norm_Int'])
        base_rgb_val = utils_for_plotly_rewrites.rgb(wavelength_start, gamma=utils_for_plotly_rewrites.gamma_factor)

        if scale_by_int is True:
            final_intensity_scale = utils_for_plotly_rewrites.final_scale(utils_for_plotly_rewrites.min_bright, int_factor)
        else:
            final_intensity_scale = 1.0

        colored_rgb = utils_for_plotly_rewrites.colored_rgb(base_rgb_val, final_intensity_scale)
        color_str = f"rgb({int(colored_rgb[0])}, {int(colored_rgb[1])}, {int(colored_rgb[2])})"

        line_fig.add_trace(go.Scatter(
            x=[wavelength_start, wavelength_end],
            y=[data_df.iloc[i][helper_utils.INT_col], data_df.iloc[i+1][helper_utils.INT_col]],
            mode='lines',
            line=dict(color=color_str, width=2), # linewidth=2 from line_plot_iteration
            showlegend=False
        ))

    line_fig.update_layout(axis_labels(graph_type='line', show_peak_labels=show_peak_labels))
    line_fig.show()


def scatter_iter(data_df, show_peak_labels, show_label_colour, scale_by_int):
    scatter_fig=go.Figure()

    if show_peak_labels is True:
        scatter_fig.update_layout(annotations = peak_labels(data_df=data_df, show_label_colour=show_label_colour, has_any_nist=False))

    alpha_factor = data_df['Norm_Int']
    base_marker_size_plotly = 2
    max_marker_size_factor_plotly = 10
    sizes = base_marker_size_plotly + (max_marker_size_factor_plotly * alpha_factor)

    if scale_by_int is True:
        alphas = utils_for_plotly_rewrites.final_scale(utils_for_plotly_rewrites.min_alpha_scatter, alpha_factor)
    else:
        alphas = 1.0

    rgba_colors = []
    for i in range(len(data_df)):
        wavelength = data_df.iloc[i][helper_utils.wl_col]
        r, g, b = utils_for_plotly_rewrites.rgb(wavelength, gamma=utils_for_plotly_rewrites.gamma_factor)
        if scale_by_int is True:
            a = alphas.iloc[i]
        else:
            a = alphas
        rgba_colors.append(f"rgba({int(r)}, {int(g)}, {int(b)}, {a})")

        #print(r,g,b)

    scatter_fig.add_trace(go.Scatter(
        x=data_df[helper_utils.wl_col],
        y=data_df[helper_utils.INT_col],
        mode='markers',
        marker=dict(
            color=rgba_colors,
            size=sizes,
            line=dict(
                width=0
            )
        ),
        showlegend=False,
    ))
    scatter_fig.update_layout(axis_labels(graph_type='scatter', show_peak_labels=show_peak_labels))
    scatter_fig.show()



def gaussian_iter(
        df_plot_data, 
        show_peak_labels,
        show_label_colour,
        detect_columns,
        nm_col,
        int_col,
        #mode,
        save_path=None,
        scale_by_int=False,
        scale_mode = None, 
        peak_wavelengths=None,
):
    gauss_fig=go.Figure()

    df_plot_data, should_exit_early, has_any_nist = prep_utils.prep_with_nist(data_df = df_plot_data, detect_columns=detect_columns, nm_col=nm_col, int_col=int_col,)
    #if should_exit_early:




    if show_peak_labels is True:
            gauss_fig.update_layout(annotations = peak_labels(data_df=df_plot_data, show_label_colour=show_label_colour, has_any_nist=has_any_nist))

    if peak_wavelengths is not None:
        nm_vals = df_plot_data[helper_utils.wl_col].values
        peaks_indices = []
        for pw in peak_wavelengths:
            try:
                pv = float(pw)
            except Exception:
                continue
            idx = int(np.argmin(np.abs(nm_vals - pv)))
            peaks_indices.append(idx)
        peaks_indices = sorted(set(peaks_indices))
    elif has_any_nist:
        peaks_indices = peaks_indices, _properties = find_peaks(df_plot_data['_adj_int'], prominence=helper_utils.DEFAULT_PROM_PERC) 


        #plt.figure(figsize=helper_utils.fig_size)


    else:
        dyn_prominence = utils_for_plotly_rewrites.dynamic_prominence(helper_utils.DEFAULT_PROM_PERC, prep_utils.int_range)
        peaks_indices, _properties = find_peaks(df_plot_data[helper_utils.INT_col], prominence=dyn_prominence)

    int_vals = df_plot_data['_raw_int'].values
    y_max = int_vals.max()

    #print("\nDEBUG:", int_vals)
    
    # Create a new, denser wavelength array for plotting the synthetic spectrum
    x_synthetic = np.linspace(400, 750, 1000) # 1000 points for a smooth synthetic curve
    y_synthetic = np.zeros_like(x_synthetic)

    for i, peak_idx in enumerate(peaks_indices):
        peak_nm = float(df_plot_data.iloc[peak_idx][helper_utils.wl_col])
        peak_amplitude = float(df_plot_data.iloc[peak_idx]['_adj_int'])
        normalized_amplitude = float(df_plot_data.iloc[peak_idx]['Norm_Int'])

        # Scale sigma based on normalized intensity (higher intensity = broader peak)
        sigma = utils_for_plotly_rewrites.base_sigma_nm + (utils_for_plotly_rewrites.max_sigma_multiplier - 1) * utils_for_plotly_rewrites.base_sigma_nm * normalized_amplitude

        # Create a Gaussian curve for this peak
        gaussian_curve = peak_amplitude * np.exp(-((x_synthetic - peak_nm)**2) / (2 * sigma**2))
        y_synthetic += gaussian_curve # Add to the total synthetic spectrum

    # Normalize the synthetic spectrum intensities for coloring
    min_y_synthetic = y_synthetic.min()
    max_y_synthetic = y_synthetic.max()
    if (max_y_synthetic - min_y_synthetic) == 0:
        normalized_y_synthetic = np.ones_like(y_synthetic)
    else:
        normalized_y_synthetic = (y_synthetic - min_y_synthetic) / (max_y_synthetic - min_y_synthetic)

    # Iterate through each segment of the synthetic spectrum to apply color and alpha
    for i in range(len(x_synthetic) - 1):
        wavelength_start = x_synthetic[i]
        wavelength_end = x_synthetic[i+1]
        y_val_start = y_synthetic[i]
        y_val_end = y_synthetic[i+1]

        base_rgb_tuple = utils_for_plotly_rewrites.rgb(wavelength_start)
        segment_alpha = utils_for_plotly_rewrites.final_scale(utils_for_plotly_rewrites.min_alpha, normalized_y_synthetic[i])

        r, g, b = int(base_rgb_tuple[0]), int(base_rgb_tuple[1]), int(base_rgb_tuple[2])
        fill_color_str = f"rgba({r}, {g}, {b}, {segment_alpha})"
        line_color_str = '#333333'

        gauss_fig.add_trace(go.Scatter(
            x=[wavelength_start, wavelength_end],
            y=[y_val_start, y_val_end],
            mode='lines',
            line=dict(color=fill_color_str, width=0), # Set width to 0 to remove segment outlines
            fill='tozeroy',
            fillcolor=fill_color_str,
            showlegend=False,
            name=f''        
        ))

    gauss_fig.update_layout(axis_labels(graph_type='gaussian', show_peak_labels=show_peak_labels))
    gauss_fig.show()

def filled_iter(data_df, show_peak_labels, show_label_colour, scale_by_int):
    filled_fig=go.Figure()

    if show_peak_labels is True:
        filled_fig.update_layout(annotations = peak_labels(data_df=data_df, show_label_colour=show_label_colour, has_any_nist=False))

    for i in range(len(data_df) - 1):
        wavelength_start = data_df.iloc[i][helper_utils.wl_col]
        wavelength_end = data_df.iloc[i+1][helper_utils.wl_col]
        y_val_start = data_df.iloc[i][helper_utils.INT_col]
        y_val_end = data_df.iloc[i+1][helper_utils.INT_col]

        base_rgb_tuple = utils_for_plotly_rewrites.rgb(wavelength_start, gamma=utils_for_plotly_rewrites.gamma_factor) 

        alpha_factor = data_df.iloc[i]['Norm_Int']

        if scale_by_int is True:
            alpha = utils_for_plotly_rewrites.final_scale(utils_for_plotly_rewrites.min_alpha, alpha_factor)
        else:
            alpha = 1.0

        r, g, b = int(base_rgb_tuple[0]), int(base_rgb_tuple[1]), int(base_rgb_tuple[2])
        fill_color_str = f"rgba({r}, {g}, {b}, {alpha})"

        #print(base_rgb_tuple)

        line_color_str = f"rgb({r}, {g}, {b})"

        filled_fig.add_trace(go.Scatter(
            x=[wavelength_start, wavelength_end],
            y=[y_val_start, y_val_end],
            mode='lines',
            line=dict(color=line_color_str, width=2), # linewidth=2 from line_plot_iteration
            fill='tozeroy', # Fill area below the line to y=0
            fillcolor=fill_color_str,
            showlegend=False,
            name=f'' # Add a name for potential debugging, though not shown
        ))

    filled_fig.update_layout(axis_labels(graph_type='filled line', show_peak_labels=show_peak_labels))
    filled_fig.show()


def bar_iter(data_df, show_peak_labels, show_label_colour, scale_by_int):
    fig_bar = go.Figure();


    if show_peak_labels is True:
        fig_bar.update_layout(annotations = peak_labels(data_df=data_df, show_label_colour=show_label_colour, has_any_nist=False))

    bar_colors = []
    for index, row in data_df.iterrows():
        wavelength = row[helper_utils.wl_col]
        normalized_intensity = row['Norm_Int']

        base_rgb=utils_for_plotly_rewrites.rgb(wavelength)

        if scale_by_int is True:
            final_intensity_scale=utils_for_plotly_rewrites.final_scale(utils_for_plotly_rewrites.min_bright, normalized_intensity)
        else:
            final_intensity_scale = 1.0

        colored_rgb=utils_for_plotly_rewrites.colored_rgb(base_rgb, final_intensity_scale)

        r, g, b = int(colored_rgb[0]), int(colored_rgb[1]), int(colored_rgb[2])
        bar_colors.append(f"rgba({r}, {g}, {b}, {1.0})")

    
    fig_bar.add_trace(go.Bar(
        x=data_df[helper_utils.wl_col],
        y=data_df[helper_utils.INT_col],
        marker=dict(color=bar_colors), # Use marker dict for color
        marker_line_color=bar_colors,
        marker_line_width=0,
        showlegend=False,
    ));

    fig_bar.update_layout(axis_labels(graph_type='bar', show_peak_labels=show_peak_labels))
    fig_bar.show()


def non_rgb_iter(data_df, show_peak_labels, show_label_colour):
    non_rgb_fig=go.Figure()


    if show_peak_labels is True:
        non_rgb_fig.update_layout(annotations = peak_labels(data_df=data_df, show_label_colour=show_label_colour, has_any_nist=False))


    non_rgb_fig = px.line(data_df, x=data_df[helper_utils.wl_col], y=data_df[helper_utils.INT_col],
        color_discrete_sequence=['#1f77b4']) # Matching Matplotlib's default blue

    peaks, _ = find_peaks(data_df[helper_utils.INT_col], prominence=utils_for_plotly_rewrites.dynamic_prominence(utils_for_plotly_rewrites.prominence, prep_utils.int_range))

    non_rgb_fig.update_layout(axis_labels(graph_type='non rgb line', show_peak_labels=show_peak_labels))
    non_rgb_fig.show()


In [18]:
def trad_spec_labels(
    has_any_nist,
    scale_mode,
    fig_height,
    title=None,
    random_title=True,
    x_title=utils_for_plotly_rewrites.X_TITLE,
    show_grid=False,
    bg_colour = utils_for_plotly_rewrites.BG,
    text_colour=utils_for_plotly_rewrites.COLOUR,
    grid_color='#333333', # Adjusted to a very dark grey
    dtick_x=50,
    ):

    if title:
        plot_title = str(title).strip()
    if random_title:
        plot_title= str(utils_for_plotly_rewrites.generate_random_title())


    layout_config = dict (
        plot_bgcolor=utils_for_plotly_rewrites.BG,
        paper_bgcolor=utils_for_plotly_rewrites.BG,
        width = utils_for_plotly_rewrites.FIG_WIDTH * 80,
        height = fig_height,
        title=dict(text=plot_title, yref='container', xref='container', yanchor='top', xanchor='center', y=0.97, x=0.5),
        margin=dict(l=10, r=10, b=30, t=10), 
        font=dict(
            color=text_colour,
            #family = 'DejaVu Sans, sans-serif',            # can only be set in .js
            size=11,
            ),
        xaxis=dict(
            range=[utils_for_plotly_rewrites.X_MIN, utils_for_plotly_rewrites.X_MAX],
            gridcolor=bg_colour,
            showgrid=False,
            title=dict(text=x_title, standoff=4),
            dtick=dtick_x,
            ticks='outside',
            tickcolor=text_colour,
            minor = dict(dtick=10, ticks='outside', tickcolor=text_colour, tickwidth=0.5),
            showline=False,
            automargin=True,
        ),
        yaxis=dict(
            range=[0,1],
            visible=False,
            automargin=True,
        )
    )

    return layout_config

    ## ax.tick_params(axis='x', which='major', colors=text_color, labelsize=10)
    ## ax.tick_params(axis='x', which='minor', colors=text_color, length=4, width=0.5)


def plot_trad(
    df_plot_data,
    scale_mode,
    detect_columns,
    nm_col,
    has_any_nist,
    show_peak_labels,
    title,
    random_title,
    show_grid = False,
    scale_by_int = None,
    save_path = None,
    prominence_percentage=0,     
    fig_size=utils_for_plotly_rewrites.FIG_SIZE,
    min_brightness=0,
    peak_wavelengths=None,
    x_min=utils_for_plotly_rewrites.X_MIN,
    x_max=utils_for_plotly_rewrites.X_MAX,
    min_needle_max_width_nm=utils_for_plotly_rewrites.MIN_NEEDLE_WIDTH,
    max_needle_max_width_nm=utils_for_plotly_rewrites.MAX_NEEDLE_WIDTH,
    needle_shape_power=utils_for_plotly_rewrites.NEEDLE_POWER_SHAPE,
    glow_width_multiplier=utils_for_plotly_rewrites.GLOW_WIDTH_MULT,
    glow_alpha=0,
    max_needle_y_scale=0,
    dpi=utils_for_plotly_rewrites.DPI,
    peak_label_y_position=0,
    label_min_norm_int=utils_for_plotly_rewrites.LABEL_NORM_INT,
    subplots_adjust_top = 0.90,
    max_needle_y = utils_for_plotly_rewrites.MAX_Y_SCALE,
    fig_height_overflow_scale = 9.0 * 80,
    fig_height_base = utils_for_plotly_rewrites.FIG_HEIGHT_BASE * 80,
):

    if show_peak_labels is False:
        max_needle_y_scale = 0.98
    else:
        max_needle_y_scale = max_needle_y

    trad_fig = go.Figure()


    #new_utils.trad_spec_labels(fig=fig, ax=ax, x_min=x_min, x_max=x_max, title=title, random_title=random_title, has_any_nist=has_any_nist, scale_mode=scale_mode)

    if has_any_nist:
        scale_mode = None
        min_brightness = utils_for_plotly_rewrites.DEFAULT_MIN_BRIGHT
        glow_alpha = utils_for_plotly_rewrites.DEFAULT_GLOW_ALPHA
        prominence_percentage = utils_for_plotly_rewrites.DEFAULT_PROM_PERC
        peak_label_y_position = max_needle_y + 0.01
    else:
        if scale_mode == 'raw':
            glow_alpha = utils_for_plotly_rewrites.DEFAULT_GLOW_ALPHA
            peak_label_y_position = utils_for_plotly_rewrites.DEFAULT_PEAK_LABEL_POSN
        elif scale_mode == 'normalize':
            glow_alpha = utils_for_plotly_rewrites.NORM_GLOW_ALPHA
            peak_label_y_position = utils_for_plotly_rewrites.NORM_PEAK_LABEL_POSN

    # --- Peak Detection ---
    if peak_wavelengths is not None:
        nm_vals = df_plot_data[helper_utils.wl_col].values
        peaks = []
        for pw in peak_wavelengths:
            try:
                pv = float(pw)
            except Exception:
                continue
            idx = int(np.argmin(np.abs(nm_vals - pv)))
            peaks.append(idx)
        peaks = sorted(set(peaks))
    elif has_any_nist:
        print ("DEBUG: has nist plot trad", has_any_nist)
        peaks = nist_codes.identify_spectral_peaks(df_plot_data.reset_index(drop=True), prominence_percentage, peak_wavelengths=peak_wavelengths)
    else:
        raw_min = df_plot_data[detection_col].min()
        raw_max = df_plot_data[detection_col].max()
        raw_range = raw_max - raw_min
        if scale_mode == 'raw':
            prominence_percentage = utils_for_plotly_rewrites.DEFAULT_PROM_PERC
        if scale_mode == 'normalize':
            prominence_percentage = utils_for_plotly_rewrites.NORM_PROM_PERC
        dynamic_prominence = prominence_percentage * (raw_range if raw_range != 0 else 1.0)
        peaks, _ = find_peaks(df_plot_data[helper_utils.INT_col], prominence=dynamic_prominence)
    
    peak_nms = [float(df_plot_data.iloc[index][helper_utils.wl_col]) for index in peaks]
    peak_ints = [float(df_plot_data.iloc[index]["Norm_Int"]) for index in peaks]
    init_peak_label_y_posn = peak_label_y_position

    if show_peak_labels is True:
        try:
            label_ys = helper_utils.compute_label_positions(
                peak_nms,
                intensities=peak_ints,
                base_y=init_peak_label_y_posn,
                min_sep_nm=0.5,
                y_step=0.06,
                method="prefer_stronger_top",
                max_y=0.98,
            )
        except Exception:
            label_ys = [init_peak_label_y_posn] * len(peaks)

        if label_ys:
            max_label_y = max(label_ys)
            overflow = max(0, max_label_y - init_peak_label_y_posn)
            
            print("\noverflow:", overflow, "\n fig height:", fig_height_base)
            print("o.g. peak label y pos'n:", init_peak_label_y_posn)
            if overflow > 0:
                new_height = fig_height_base + overflow * fig_height_overflow_scale
                new_fig_height = new_height
                current_needle_height_in = max_needle_y * new_height
                new_needle_height = (max_needle_y * fig_height_base) / new_height 
                max_needle_y_scale = new_needle_height
                new_peak_y_posn = (init_peak_label_y_posn * fig_height_base) / new_height
                peak_label_y_position = new_peak_y_posn

                print("\nog max needle y:", max_needle_y, "\nmax label y:", max_label_y, "\ninit peak label y:", init_peak_label_y_posn)
                print("needle height goal:", new_needle_height, "\n \t inches:", new_needle_height*new_height)
                print("current needle height (in):", current_needle_height_in)
                print("height:", new_height, "max label y:", max_label_y)
                print("\npeak label y pos'n goal:", new_peak_y_posn, "\nnew fig height:", new_fig_height,)

            else:
                print("Labels fit — no expansion needed")
                max_needle_y_scale = max_needle_y + 0.1
                peak_label_y_position = peak_label_y_position + 0.08
                print("peak label y pos'n:", peak_label_y_position)
        else:
            max_needle_y_scale = max_needle_y
            print("No labels on this spectrum")
        print(f"DEBUG: max_label_y = {max(label_ys) if label_ys else 'N/A'}")

        try:
            label_ys = helper_utils.compute_label_positions(
                peak_nms,
                intensities=peak_ints,
                base_y=peak_label_y_position,   # now uses updated value
                min_sep_nm=0.9,
                y_step=0.07,
                method="prefer_stronger_top",
                max_y=0.98,
            )
        except Exception:
            label_ys = [peak_label_y_position] * len(peaks)
        print("\n new peak label pos'n:", peak_label_y_position)

    # Render every transition as a faint needle (increase visibility for verification)
    _bg_y_top = max_needle_y_scale  # use full height for visibility
    print("\n bg y top", _bg_y_top)
    _bg_y = np.linspace(0, _bg_y_top, 40)
    for _idx, _row in df_plot_data.iterrows():
        _nm = float(_row[nm_col])
        _ni = float(_row.get('Norm_Int', 0.0))
        _base_rgb = utils_for_plotly_rewrites.rgb(_nm)
        _final_scale = utils_for_plotly_rewrites.final_scale(min_brightness, _ni)
        _color = (_base_rgb[0] * _final_scale, _base_rgb[1] * _final_scale, _base_rgb[2] * _final_scale)
        _width_mult = float(_row.get('_width_mult', 1.0))
        _colour = f"rgb({int(_color[0])}, {int(_color[1])}, {int(_color[2])})"
        _bg_base_width = min_needle_max_width_nm * 1.0 * _width_mult   # make background lines thicker for test
        _bg_widths = _bg_base_width * np.ones_like(_bg_y)
        _x_left = utils_for_plotly_rewrites.left_x(_nm, _bg_widths)
        _x_right = utils_for_plotly_rewrites.right_x(_nm, _bg_widths)
        #ax.fill_betweenx(_bg_y, _x_left, _x_right, facecolor=_color, alpha=glow_alpha, edgecolor='none', linewidth=0, zorder=0)

        loop_x = list(_x_left) + list(_x_right)[::-1]
        loop_y = list(_bg_y) + list(_bg_y)[::-1]

        trad_fig.add_trace(go.Scatter(
            x=loop_x,
            y=loop_y,
            fill='toself',
            fillcolor=_colour,
            opacity=glow_alpha,
            line=dict(width=0),      # Equivalent to edgecolor='none', linewidth=0
            mode='none',             # Hides individual markers/lines, showing only the fill
            showlegend=False,
            hoverinfo='skip'         # Keeps background elements from triggering tooltips
        ))

    # Plot sharp distinct lines for each identified peak using fill_betweenx for needle shape
    for j, peak_index in enumerate(peaks):
        peak_nm = df_plot_data.iloc[peak_index][nm_col]
        peak_norm_int = df_plot_data.iloc[peak_index]['Norm_Int']
        base_rgb = utils_for_plotly_rewrites.rgb(peak_nm)

        # Emphasize peaks: gentle gamma + emphasis multiplier for labeled peaks
        _peak_gamma = 0.8
        if has_any_nist or scale_mode == 'raw':   
            _peak_emphasis = utils_for_plotly_rewrites.DEFAULT_PEAK_EMPHASIS
            min_brightness = utils_for_plotly_rewrites.DEFAULT_MIN_BRIGHT
        elif scale_mode == 'normalize':
            _peak_emphasis = utils_for_plotly_rewrites.NORM_PEAK_EMPHASIS
            min_brightness = utils_for_plotly_rewrites.NORM_MIN_BRIGHT
        pre_int_scale = min_brightness + (1 - min_brightness) * (peak_norm_int ** _peak_gamma)
        final_int_scale = min(1.0, pre_int_scale * _peak_emphasis)
        color_rgb = utils_for_plotly_rewrites.colored_rgb(base_rgb, final_int_scale)

        # Scale the maximum width of the needle based on normalized intensity
        base_width = min_needle_max_width_nm + (max_needle_max_width_nm - min_needle_max_width_nm) * peak_norm_int
        width_mult = df_plot_data.iloc[peak_index].get('_width_mult', 1.0)

        # boost peak widths so labeled peaks stand out
        _peak_width_boost = 1.6
        scaled_max_width_nm = base_width * width_mult * _peak_width_boost
        
        # Calculate the actual height the needle should reach (fraction of 0-1)
        current_peak_render_height = max_needle_y_scale

        # Define y-coordinates for the needle shape, spanning from 0 to current_peak_render_height
        y_coords_render = np.linspace(0, current_peak_render_height, 120)
        # Calculate normalized y-coordinates for the width profile
        y_coords_normalized_for_width = (y_coords_render / current_peak_render_height if current_peak_render_height > 0 else np.zeros_like(y_coords_render))

        # Use a power-law profile for the width, making tips slimmer
        width_profile_factor = (4 * y_coords_normalized_for_width * (1 - y_coords_normalized_for_width)) ** needle_shape_power

        # --- Plot the GLOW effect first ---
        glow_current_widths_nm = scaled_max_width_nm * glow_width_multiplier * width_profile_factor
        glow_x_left = utils_for_plotly_rewrites.left_x(peak_nm, glow_current_widths_nm)
        glow_x_right = utils_for_plotly_rewrites.right_x(peak_nm, glow_current_widths_nm)


        loop_x = np.concatenate([glow_x_left, glow_x_right[::-1]])
        loop_y = np.concatenate([y_coords_render, y_coords_render[::-1]])

        color_str = f"rgb({color_rgb[0]},{color_rgb[1]},{color_rgb[2]})" 

        # 3. Add the fill trace to the figure
        trad_fig.add_trace(go.Scatter(
            x=loop_x,
            y=loop_y,
            fill='toself',
            fillcolor=color_str,
            opacity=glow_alpha,
            line=dict(width=0),      # Replaces edgecolor='none' and linewidth=0
            mode='none',             # Hides the bounding lines completely
            showlegend=False,
            hoverinfo='skip'         # Prevents background element hover tooltips
        ))
                # --- Plot the main NEEDLE on top of the glow ---
        current_widths_nm = utils_for_plotly_rewrites.dynamic_prominence(scaled_max_width_nm, width_profile_factor)
        x_left = utils_for_plotly_rewrites.left_x(peak_nm, current_widths_nm)
        x_right = utils_for_plotly_rewrites.right_x(peak_nm, current_widths_nm)
        loop_x = np.concatenate([x_left, x_right[::-1]])
        loop_y = np.concatenate([y_coords_render, y_coords_render[::-1]])



        # 3. Add to the figure (execute this AFTER the zorder=1 trace)
        trad_fig.add_trace(go.Scatter(
            x=loop_x,
            y=loop_y,
            fill='toself',
            fillcolor=color_str,
            #opacity=glow_alpha,
            line=dict(width=0),      # Replaces edgecolor='none' and linewidth=0
            mode='none',             # Smooth fill with no outline paths
            showlegend=False,
            hoverinfo='skip'         # Keeps hover data focused on foreground lines
        ))

        # Add text label for the peak wavelength with rotation and stroke for readability
        if show_peak_labels and peak_norm_int >= label_min_norm_int:
            y_for_label = label_ys[j] if j < len(label_ys) else peak_label_y_position
            
            trad_fig.add_annotation(
                x=peak_nm,
                y=y_for_label,
                text=f"{peak_nm:.2f}",
                font=dict(
                    family='DejaVu Sans, sans-serif',
                    #weight=100,
                    color="white",
                    size=10.5  # Matplotlib size 8 translates roughly to 10-11px in Plotly
                ),
                showarrow=False,
                textangle=-60,      # Plotly rotates clockwise; -60 equivalent to +60 counter-clockwise
                xanchor="center",     
                yanchor="middle",   

                xshift=5,
                yshift=25,
                
                # Replicates path_effects=[pe.withStroke(linewidth=1.5, foreground="black")]
                # Uses CSS text-shadow to create a crisp 1.5px black outline
                captureevents=False # Ensures annotation doesn't block plot clicks
            )
            
            # Apply the text stroke using Plotly's HTML template support
            # This edits the last added annotation directly to inject the CSS shadow style
            #trad_fig.layout.annotations[-1].font.color = "white"
            #trad_fig.layout.annotations[-1].text = (
            #    f"<span style='text-shadow: "
            #    f"-1.5px -1.5px 0 #000, 1.5px -1.5px 0 #000, "
            #    f"-1.5px 1.5px 0 #000, 1.5px 1.5px 0 #000;'>"
            #    f"{peak_nm:.2f}</span>"
            #)


    #plt.tight_layout(rect=[0, 0, 1, 0.92])


    if new_fig_height > fig_height_base:
        fig_height = new_fig_height
    else:
        fig_height = fig_height_base

    trad_fig.update_layout(
        trad_spec_labels(has_any_nist=has_any_nist, scale_mode=scale_mode, fig_height=fig_height))


    return trad_fig


def trad_spec(
    data_df, 
    #mode,
    scale_mode='raw',
    show_peak_labels=True,
    nm_col=None,
    int_col=None,
    detect_columns=False,
    title = None,
    random_title = None,
    show_grid = False,
    show_label_colour = None,
    scale_by_int = None,
    fig_size=utils_for_plotly_rewrites.FIG_SIZE, 
    dpi=utils_for_plotly_rewrites.DPI,
):
    df_plot_data, should_exit_early, has_any_nist = prep_utils.prep_with_nist(data_df = data_df, nm_col=nm_col, int_col=int_col, detect_columns=detect_columns)


    #if should_exit_early:

    FIG = go.Figure()
    FIG = plot_trad(
        df_plot_data = df_plot_data, 
        nm_col = helper_utils.wl_col, 
        has_any_nist=has_any_nist,
        scale_mode=scale_mode,
        show_peak_labels=show_peak_labels,
        detect_columns=detect_columns,
        title=title,
        random_title=random_title,
    )

    #FIG.update_layout(trad_spec_labels(has_any_nist=has_any_nist, scale_mode=scale_mode))

    config = {
        'toImageButtonOptions': {
            'format': 'png',
            'scale': 10.0,
            'filename': f"{utils_for_plotly_rewrites.generate_random_title()}" 
        }
    }

    FIG.show(config=config)

In [19]:
def get_graph_type():
    global graph_type, rgb_type
    RGB_TYPE = input("Render as RGB Spectrum? Choose: Yes or No").lower()
    if RGB_TYPE == 'yes' or RGB_TYPE == 'y':
        rgb_type = 'yes'
        GRAPH_TYPE = input("Choose Graph Type: Line, Bar, Scatter, Gaussian, Traditional").lower()
        if GRAPH_TYPE == 'bar' or GRAPH_TYPE == 'b':
            graph_type = 'bar'
        elif GRAPH_TYPE == 'scatter' or GRAPH_TYPE == 's':
            graph_type = 'scatter'
        elif GRAPH_TYPE == 'gaussian' or GRAPH_TYPE == 'g':
            graph_type = 'gaussian'
        elif GRAPH_TYPE == 'line' or GRAPH_TYPE == 'l':
            FILL_TYPE = input("Filled Graph? Choose: Yes or No").lower()
            if FILL_TYPE == 'yes' or FILL_TYPE == 'y':
                graph_type = 'filled line'
            else:
                graph_type = 'line'           
        elif GRAPH_TYPE == 'traditional' or GRAPH_TYPE == 'trad' or GRAPH_TYPE == 't':
            graph_type = 'traditional'
        return GRAPH_TYPE
    elif RGB_TYPE == 'no' or RGB_TYPE == 'n':
        rgb_type = 'no'
        graph_type = 'non rgb line'

    print("\ngraph type:", graph_type)

    return RGB_TYPE


def scale_by_int_():
    global SCALE_BY_INT
    scale = input("Scale brightness by intensity? Yes or No").lower()
    if scale == 'yes' or scale == 'y':
        SCALE_BY_INT = 'yes'
    elif scale == 'no' or scale == 'n':
        SCALE_BY_INT = 'no'
    print('\nScale brightness:', SCALE_BY_INT.capitalize())


def show_peak_labels_():
    global SHOW_PEAKS, show_rgb_peaks
    show_labels = input("Show Peak Labels? Yes or No").lower()
    if show_labels == 'yes' or show_labels == 'y':
        SHOW_PEAKS = 'yes'
        show_rgb = input("Show Peak Labels in Colour? Yes or No").lower()
        if show_rgb == 'yes' or show_rgb == 'y':
            show_rgb_peaks = 'yes'
        if show_rgb == 'no' or show_rgb == 'n':
            show_rgb_peaks = 'no'
    if show_labels == 'no' or show_labels == 'n':
        SHOW_PEAKS = 'no'


def get_generic_type():
    global generic_type
    gen_type = (input('Choose Rendering: Raw or Normalised')).lower()
    if gen_type == 'raw' or gen_type == 'r':
        generic_type = 'raw'
    elif gen_type == 'normalised' or gen_type == 'norm' or gen_type == 'n':
        generic_type = 'normalize'
    print ('Rendering Type: ', generic_type)
    return generic_type


In [20]:
def plot_other_spec(
    data_df,
    detect_columns,
    nm_col,
    int_col,
    reverse_x,
    force_nist = None,
    #graph_type = None,
    scale_mode = 'auto',
    title = None,
    random_title = None,
    show_peak_labels = None,
    show_grid = True,
    show_label_colour = None,
    scale_by_int = None,
    save_path=None,
):

    print ("DEBUG: plot_other_spec")

    get_graph_type()

    show_peak_labels_()
    if SHOW_PEAKS == 'yes':
        show_peak_labels = True
    elif SHOW_PEAKS == 'no':
        show_peak_labels = False

    if graph_type == 'traditional':
        get_generic_type()
        trad_spec(data_df=data_df, detect_columns=detect_columns, scale_mode=generic_type, show_peak_labels=show_peak_labels, title=title, random_title=random_title, nm_col=nm_col, int_col=int_col)
    else:
        data_df, should_exit_early = prep_utils.prep_other(data_df = data_df, detect_columns=detect_columns, nm_col=nm_col, int_col=int_col)
        #if should_exit_early:

        if rgb_type == 'yes':
            scale_by_int_()
            if SCALE_BY_INT == 'yes':
                scale_by_int = True
            elif SCALE_BY_INT == 'no':
                scale_by_int = False

        if graph_type == 'bar':
            bar_iter(data_df = data_df, show_peak_labels=show_peak_labels, show_label_colour=show_label_colour, scale_by_int=scale_by_int)
        elif graph_type == 'scatter':
            scatter_iter(data_df = data_df, show_peak_labels=show_peak_labels, show_label_colour=show_label_colour, scale_by_int=scale_by_int)
        elif graph_type == 'gaussian':
            gaussian_iter(df_plot_data=data_df, detect_columns=detect_columns,show_peak_labels=show_peak_labels, show_label_colour=show_label_colour, int_col=int_col, nm_col=nm_col, scale_by_int=False)
        elif graph_type == 'line':
            line_iter(data_df = data_df, show_peak_labels=show_peak_labels, show_label_colour=show_label_colour, scale_by_int=scale_by_int)
        elif graph_type == 'filled line':
            filled_iter(data_df = data_df, show_peak_labels=show_peak_labels, show_label_colour=show_label_colour, scale_by_int=scale_by_int)
        elif graph_type == 'non rgb line':
            non_rgb_iter(data_df = data_df, show_peak_labels=show_peak_labels, show_label_colour=show_label_colour)


In [21]:
def detect_prep(
    data_df,
    detect_columns = None,
    nm_col = None,
    int_col = None,
    reverse_x = None,
    force_nist = None,
    #graph_type = None,
    scale_mode = 'auto',
    title = None,
    random_title = None,
    show_peak_labels = None,
    show_grid = True,
    show_label_colour = None,
    scale_by_int = None,
    save_path=None,
):
    df_data, has_any_nist, _, _, _ = prep_utils.run_nist_check(data_df=data_df, detect_columns=detect_columns, force_nist=force_nist, nm_col=nm_col, int_col=int_col)

    if has_any_nist is True:
        prep_type = 'nist'
        nist_plot_type = input("NIST Descriptors Detected. Choose NIST Graph Type: Gaussian or Traditional").lower()
        if nist_plot_type == 'gaussian' or nist_plot_type == 'g':
            graph_type = 'gaussian'
            gaussian_iter(df_plot_data=df_data, detect_columns=detect_columns,show_peak_labels=show_peak_labels, show_label_colour=show_label_colour, int_col=int_col, nm_col=nm_col)
        if nist_plot_type == 'traditional' or nist_plot_type == 'trad' or nist_plot_type == 't':
            scale_mode = 'raw'
            graph_type = 'traditional'
            trad_spec(data_df=df_data, detect_columns=detect_columns, scale_mode=scale_mode, show_peak_labels=show_peak_labels, title=title, random_title=random_title, nm_col=nm_col, int_col=int_col)
    else:
        prep_type = 'generic'
        plot_other_spec(data_df=df_data, detect_columns=detect_columns, nm_col=nm_col, int_col=int_col, reverse_x=reverse_x)
    #return prep_type, graph_type, scale_mode

In [22]:
HE_DF = pd.read_excel(HE_SHEET)
NIST_DF = pd.read_excel(NIST_SHEET)
H_DF = pd.read_excel(H_SHEET)
HE_INC_INT_DF = pd.read_excel(HE_INC_INT_SHEET)

In [23]:
#detect_prep(HE_DF)

In [24]:
def plot(
    data_df,
    detect_columns=None,
    nm_col=None,
    int_col=None,
    reverse_x=None,
    force_nist = None,
    graph_type = None,
    scale_mode = 'auto',
    title = None,
    random_title = None,
    show_peak_labels = True,
    show_grid = True,
    show_label_colour = True,
    scale_by_int = True,
    save_path=None,
):
    data_df, should_exit_early = prep_utils.prep_other(data_df = data_df, detect_columns=detect_columns, nm_col=nm_col, int_col=int_col)
    scatter_iter(data_df = data_df, show_peak_labels=show_peak_labels, show_label_colour=show_label_colour, scale_by_int=scale_by_int)

In [29]:
data_df, should_exit_early, has_any_nist = prep_utils.prep_with_nist(data_df = NIST_DF, detect_columns=None, nm_col=None, int_col=None)

gaussian_iter(df_plot_data=data_df, detect_columns=None, nm_col=None, int_col=None, show_peak_labels=True, show_label_colour=True, scale_by_int=False)

NIST Destriptors Detected. Preparing NIST Rendering.
(prep_with_nist) ymax =  490.0
NIST Destriptors Detected. Preparing NIST Rendering.
(prep_with_nist) ymax =  490.0
(set_y_lim) 500


data_df, should_exit_early, _, = prep_utils.prep_with_nist(data_df = NIST_DF, detect_columns=None, nm_col=None, int_col=None)

trad_spec(data_df=data_df)